# 6.1 — Perceptron & Multilayer Perceptrons

Perceptrons and multilayer perceptrons (MLPs) turn vectors into predictions by repeatedly applying an affine score, a nonlinear gate, and a small gradient-driven update. In this lesson, you will build each piece from scratch in NumPy, inspect every intermediate value, and see why shape, scale, nonlinearity, probability, gradients, and memory bookkeeping all matter.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build perceptrons and MLPs one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is shown so the network is a composed set of visible operations, not a black box. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vector products, exponentials, and gradient arithmetic.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any random toy data.

### 1. The perceptron affine score

A perceptron begins with an **affine score**: multiply each input by a learned weight, add the pieces, then add a bias. For one input vector $x=[1.5,-0.5]$, weights $w=[1.2,-0.4]$, and bias $b=0.4$, the score is

$$z=w^\top x+b=1.2\cdot1.5+(-0.4)\cdot(-0.5)+0.4=2.4.$$

The dot product is the evidence from features, and the bias is the threshold shift. Without the bias, the boundary must pass through the origin; with it, the model can move the decision line to match the data.

In [ ]:
x_w = np.array([1.5, -0.5])          # one two-feature input.
w_w = np.array([1.2, -0.4])          # one weight per input feature.
b_w = 0.400                          # bias shifts the threshold.
parts_w = w_w * x_w                  # per-feature evidence before summing.
print("per-feature products:", np.round(parts_w, 3))  # [1.8, 0.2].
print("bias:", b_w)                  # threshold shift.
assert np.allclose(parts_w, [1.8, 0.2])

▶ What you'll see: the positive first feature and negative second feature both contribute positive evidence here.

In [ ]:
z_w = float(np.dot(w_w, x_w) + b_w)   # affine score w^T x + b.
print("affine score z:", round(z_w, 3))
assert round(z_w, 3) == 2.400

▶ What you'll see: the scratch arithmetic gives the lesson score `2.400`.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["w0*x0", "w1*x1", "bias"], [parts_w[0], parts_w[1], b_w], color=["teal", "teal", "gray"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("1: affine score pieces")
plt.ylabel("contribution to z")
plt.show()

▶ What you'll see: three positive bars that sum to the score 2.4.

*Why it's done this way: the affine form is the simplest trainable rule that preserves linear-algebra structure — weights choose feature importance, the dot product adds evidence, and the bias moves the decision threshold without changing feature weights.*

### 2. A nonlinear gate turns scores into hidden activations

If we stack affine maps without a nonlinear function between them, the whole stack collapses into one bigger affine map. A ReLU gate, $\phi(z)=\max(0,z)$, prevents that collapse: positive scores pass forward, negative scores are clipped to zero. The clip is simple, but it changes the function class from one straight boundary to piecewise-linear shapes.

In [ ]:
z_values_w = np.array([-2.0, -0.4, 0.0, 1.1, 2.4])  # possible affine scores.
h_values_w = np.maximum(0, z_values_w)              # ReLU activation.
print("z:", z_values_w)
print("ReLU(z):", h_values_w)
assert h_values_w[-1] == 2.4 and h_values_w[0] == 0.0

▶ What you'll see: negative scores become 0, while the lesson score 2.4 passes through unchanged.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(z_values_w, h_values_w, marker="o", color="seagreen")
plt.axhline(0, color="black", linewidth=0.7)
plt.axvline(0, color="black", linewidth=0.7)
plt.title("2: ReLU gate")
plt.xlabel("affine score z")
plt.ylabel("hidden activation h")
plt.show()

▶ What you'll see: a flat left side and a sloped right side — the piecewise-linear gate.

In [ ]:
h_lesson_w = float(np.maximum(0, z_w))  # apply the gate to the score from concept 1.
print("gated lesson signal h:", round(h_lesson_w, 3))
assert round(h_lesson_w, 3) == 2.400

▶ What you'll see: because 2.4 is positive, the gate preserves the signal.

*Why it's done this way: nonlinearity is what makes depth meaningful; ReLU keeps positive evidence and blocks negative evidence, so later layers receive a shaped signal rather than another equivalent linear score.*

### 3. From one perceptron to one hidden layer

An MLP repeats the same affine-and-gate pattern for several hidden units at once. A weight matrix $W$ has one row per hidden unit, so $z=Wx+b$ computes many perceptrons in a single matrix-vector product. The output layer then combines the hidden activations into a final score.

In [ ]:
W1_w = np.array([[1.2, -0.4], [-0.7, 1.0], [0.5, 0.5]])  # three hidden perceptrons.
b1_w = np.array([0.4, 0.1, -0.2])                        # one bias per hidden unit.
z1_w = W1_w @ x_w + b1_w                                  # hidden affine scores.
print("hidden z:", np.round(z1_w, 3))
assert np.allclose(np.round(z1_w, 3), [2.4, -1.45, 0.3])

▶ What you'll see: the first hidden unit reproduces the lesson score, while the second is negative.

In [ ]:
h1_w = np.maximum(0, z1_w)                    # ReLU each hidden score independently.
W2_w = np.array([0.8, -0.3, 0.5])             # output weights combine hidden activations.
b2_w = -0.1                                   # output bias.
yhat_w = float(W2_w @ h1_w + b2_w)            # final scalar network score.
print("hidden h:", np.round(h1_w, 3))
print("output score y_hat:", round(yhat_w, 3))
assert round(yhat_w, 3) == 1.970

▶ What you'll see: the negative hidden score is clipped out, and the output becomes `1.970`.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["h0", "h1", "h2"], h1_w, color="darkorange")
plt.title("3: hidden activations passed to output")
plt.ylabel("activation")
plt.show()

▶ What you'll see: only active hidden units can influence the output score.

*Why it's done this way: a hidden layer lets the model learn reusable intermediate features; the matrix form keeps shapes explicit while the output layer decides how much each activated feature matters for the final prediction.*

### 4. Softmax turns scores into comparisons

A neural network often produces scores, but classification needs a comparison across classes. Softmax exponentiates each score and divides by the sum of all exponentials:

$$p_k=\frac{e^{s_k}}{\sum_j e^{s_j}}.$$

For scores $[2.4,0.4]$, the first class probability is $e^{2.4}/(e^{2.4}+e^{0.4})=0.881$. The subtraction trick below keeps the exponentials numerically stable without changing the probability.

In [ ]:
scores_w = np.array([2.4, 0.4])                    # two class scores.
exp_raw_w = np.exp(scores_w)                       # raw exponentials.
prob_raw_w = exp_raw_w / np.sum(exp_raw_w)         # softmax probabilities.
print("exp scores:", np.round(exp_raw_w, 3))
print("softmax:", np.round(prob_raw_w, 3))
assert np.allclose(np.round(exp_raw_w, 3), [11.023, 1.492])
assert round(float(prob_raw_w[0]), 3) == 0.881

▶ What you'll see: the 2.4 score becomes about an 88.1% probability after comparison to the 0.4 baseline.

In [ ]:
stable_scores_w = scores_w - np.max(scores_w)       # subtract max; probabilities are unchanged.
prob_stable_w = np.exp(stable_scores_w) / np.sum(np.exp(stable_scores_w))
print("stable shifted scores:", stable_scores_w)
print("same softmax:", np.round(prob_stable_w, 3))
assert np.allclose(prob_raw_w, prob_stable_w)

▶ What you'll see: shifting both scores changes exponentials but not the final probabilities.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["class 0", "class 1"], prob_stable_w, color="slateblue")
plt.ylim(0, 1)
plt.title("4: softmax probabilities")
plt.ylabel("probability")
plt.show()

▶ What you'll see: class 0 dominates because its score is two units larger.

*Why it's done this way: exponentials turn score gaps into positive relative weights, and the division normalizes those weights into probabilities; subtracting the maximum preserves the comparison while preventing overflow in larger networks.*

### 5. Loss gradients and one parameter update

Training changes parameters by small reliable nudges. For a scalar parameter $\theta=2.0$, learning rate $\eta=0.06$, and gradient $g=1.35$, gradient descent updates

$$\theta' = \theta - \eta g = 2.0 - 0.06\cdot1.35 = 1.919.$$

The minus sign matters: a positive gradient means increasing the parameter would increase loss, so descent moves the parameter downward.

In [ ]:
theta_w = 2.000              # one scalar parameter.
eta_w = 0.060                # learning rate.
grad_w = 1.350               # derivative of loss with respect to theta.
step_w = eta_w * grad_w      # amount subtracted from theta.
print("step size eta*g:", round(step_w, 3))
assert round(step_w, 3) == 0.081

▶ What you'll see: the learning rate scales the raw gradient into a smaller movement.

In [ ]:
theta_new_w = theta_w - step_w
print("theta before -> after:", round(theta_w, 3), "->", round(theta_new_w, 3))
assert round(theta_new_w, 3) == 1.919

▶ What you'll see: the parameter moves from 2.000 to 1.919, matching the lesson arithmetic.

In [ ]:
steps_axis_w = np.arange(6)
theta_path_w = theta_w - steps_axis_w * step_w
plt.figure(figsize=(4.6, 3))
plt.plot(steps_axis_w, theta_path_w, marker="o", color="crimson")
plt.title("5: repeated small descent nudges")
plt.xlabel("update number")
plt.ylabel("theta")
plt.show()

▶ What you'll see: repeated equal gradients would move the parameter steadily, not in one giant jump.

*Why it's done this way: gradients are local slope measurements, so the learning rate must make the move cautious enough that many noisy minibatch steps can accumulate into useful global learning.*

### 6. Scale, normalization, and memory bookkeeping

Deep networks are not just formulas; they are numerical systems. If an activation has mean $1.0$ and variance $0.25$, the normalized value of $2.4$ is

$$\frac{2.4-1.0}{\sqrt{0.25+10^{-5}}}=2.8.$$

That number says the activation is 2.8 standard deviations above the local mean. Meanwhile, storing 3 vectors of length 128 in 32-bit floats costs $3\cdot128\cdot4/1024=1.5$ KB. The arithmetic is tiny here, but the same terms multiplied by batches and layers become practical training limits.

In [ ]:
value_w = 2.400
mean_w = 1.000
var_w = 0.250
eps_w = 0.00001
normed_w = (value_w - mean_w) / np.sqrt(var_w + eps_w)
print("normalized value:", round(normed_w, 3))
assert round(normed_w, 3) == 2.800

▶ What you'll see: the score 2.4 is high relative to this mean and variance.

In [ ]:
vectors_w = 3
length_w = 128
bytes_per_float_w = 4
memory_kb_w = vectors_w * length_w * bytes_per_float_w / 1024
print("activation memory KB:", round(memory_kb_w, 3))
assert round(memory_kb_w, 3) == 1.500

▶ What you'll see: three 128-length float32 vectors occupy 1.5 KB.

In [ ]:
batch_sizes_w = np.array([1, 16, 64, 256])
mem_by_batch_w = batch_sizes_w * memory_kb_w
plt.figure(figsize=(4.8, 3))
plt.plot(batch_sizes_w, mem_by_batch_w, marker="o", color="purple")
plt.title("6: activation memory scales with batch")
plt.xlabel("batch size")
plt.ylabel("KB for this tiny block")
plt.show()

▶ What you'll see: memory grows linearly with batch size even for a tiny activation block.

*Why it's done this way: normalization keeps signal scale in a range where gradients are usable, and memory bookkeeping makes the hidden cost of saving activations for backprop explicit.*

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each toy uses
> small NumPy arrays, prints every intermediate with a real `# ->` value, draws one picture, and
> ends with an `assert` that pins the result.

### ✍️ Toy 1 · Affine score adds weighted evidence and bias

A perceptron score is a weighted sum plus a bias. Watch six feature contributions add up before the
bias shifts the final score.

In [ ]:
import numpy as np                              # arrays and vector arithmetic.
import matplotlib.pyplot as plt                 # one picture per toy.

t1_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t1_x = np.array([2.0, -1.0, 0.5, 3.0, -2.0, 1.0])
t1_w = np.array([1.0, -2.0, 0.5, -0.5, -1.0, 0.25])
t1_b = -0.75
t1_parts = t1_w * t1_x                          # -> [2.0, 2.0, 0.25, -1.5, 2.0, 0.25]
print("weighted parts:", t1_parts.tolist())     # -> [2.0, 2.0, 0.25, -1.5, 2.0, 0.25]
t1_sum = t1_parts.sum()                         # -> 5.0
print("sum of parts:", float(t1_sum))           # -> 5.0
t1_score = t1_sum + t1_b                        # -> 4.25
print("affine score:", float(t1_score))         # -> 4.25
assert abs(t1_score - 4.25) < 1e-12

plt.figure(figsize=(4.8, 2.8))
plt.bar(range(t1_parts.size), t1_parts, color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Toy 1 · weighted pieces before bias")
plt.xlabel("feature index")
plt.ylabel("w_i x_i")
plt.show()

▶ What you'll see: six signed feature contributions whose sum is `5.0`, then the bias lowers the score to `4.25`.

### ✍️ Toy 2 · ReLU gates negative scores

ReLU is a hard gate: negative scores become zero, while positive scores keep their value.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t2_z = np.array([-3.0, -1.0, 0.0, 0.5, 2.0, 4.0])
print("raw scores:", t2_z.tolist())             # -> [-3.0, -1.0, 0.0, 0.5, 2.0, 4.0]
t2_h = np.maximum(0.0, t2_z)                    # -> [0.0, 0.0, 0.0, 0.5, 2.0, 4.0]
print("ReLU activations:", t2_h.tolist())       # -> [0.0, 0.0, 0.0, 0.5, 2.0, 4.0]
t2_mask = (t2_z > 0).astype(int)                # -> [0, 0, 0, 1, 1, 1]
print("open-gate mask:", t2_mask.tolist())      # -> [0, 0, 0, 1, 1, 1]
t2_active_count = t2_mask.sum()                 # -> 3
print("active units:", int(t2_active_count))    # -> 3
assert int(t2_active_count) == 3

plt.figure(figsize=(4.8, 2.8))
plt.bar(range(t2_z.size), t2_h, color=["gray" if m == 0 else "seagreen" for m in t2_mask])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Toy 2 · ReLU keeps only open gates")
plt.xlabel("unit")
plt.ylabel("activation")
plt.show()

▶ What you'll see: the first three bars are zeroed out, and exactly three positive gates remain open.

### ✍️ Toy 3 · Hidden layer stacks affine gates before an output

A one-hidden-layer MLP computes several affine scores, gates them, then combines the active hidden
features into one output score.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t3_x = np.array([1.0, -2.0])
t3_W1 = np.array([[1.0, -0.5, 0.25], [0.5, 1.0, -1.5]])
t3_b1 = np.array([0.0, 0.5, -0.5])
t3_z1 = t3_x @ t3_W1                           # -> [0.0, -2.5, 3.25]
print("x @ W1:", t3_z1.tolist())               # -> [0.0, -2.5, 3.25]
t3_z1 = t3_z1 + t3_b1                          # -> [0.0, -2.0, 2.75]
print("hidden z:", t3_z1.tolist())             # -> [0.0, -2.0, 2.75]
t3_h1 = np.maximum(0.0, t3_z1)                  # -> [0.0, 0.0, 2.75]
print("hidden h:", t3_h1.tolist())             # -> [0.0, 0.0, 2.75]
t3_w2 = np.array([1.0, -1.0, 0.5])
t3_out_parts = t3_h1 * t3_w2                    # -> [0.0, -0.0, 1.375]
print("output parts:", t3_out_parts.tolist())  # -> [0.0, -0.0, 1.375]
t3_score = t3_out_parts.sum()                   # -> 1.375
print("output score:", float(t3_score))         # -> 1.375
assert abs(t3_score - 1.375) < 1e-12

plt.figure(figsize=(4.6, 2.8))
plt.bar(["h0", "h1", "h2"], t3_h1, color="darkorange")
plt.title("Toy 3 · only active hidden units contribute")
plt.ylabel("hidden activation")
plt.show()

▶ What you'll see: two hidden units are shut off, so the final score comes entirely from the third unit.

### ✍️ Toy 4 · Stable softmax preserves class probabilities

Softmax depends on score differences. Subtracting the maximum score changes the exponentials but not
the normalized probabilities.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t4_scores = np.array([3.0, 1.0, -1.0])
print("raw scores:", t4_scores.tolist())        # -> [3.0, 1.0, -1.0]
t4_exp = np.exp(t4_scores)                      # -> [20.086, 2.718, 0.368]
print("raw exp:", np.round(t4_exp, 3).tolist()) # -> [20.086, 2.718, 0.368]
t4_probs = t4_exp / t4_exp.sum()                # -> [0.867, 0.117, 0.016]
print("raw softmax:", np.round(t4_probs, 3).tolist())  # -> [0.867, 0.117, 0.016]
t4_shifted = t4_scores - t4_scores.max()        # -> [0.0, -2.0, -4.0]
print("shifted scores:", t4_shifted.tolist())   # -> [0.0, -2.0, -4.0]
t4_exp_shifted = np.exp(t4_shifted)             # -> [1.0, 0.135, 0.018]
print("shifted exp:", np.round(t4_exp_shifted, 3).tolist())  # -> [1.0, 0.135, 0.018]
t4_probs_shifted = t4_exp_shifted / t4_exp_shifted.sum()      # -> [0.867, 0.117, 0.016]
print("stable softmax:", np.round(t4_probs_shifted, 3).tolist())  # -> [0.867, 0.117, 0.016]
assert np.allclose(t4_probs, t4_probs_shifted)

plt.figure(figsize=(4.4, 2.8))
plt.bar(["class 0", "class 1", "class 2"], t4_probs_shifted, color="slateblue")
plt.ylim(0, 1)
plt.title("Toy 4 · probabilities after stable shift")
plt.ylabel("probability")
plt.show()

▶ What you'll see: the shifted logits are safer, but the class probabilities stay `[0.867, 0.117, 0.016]`.

### ✍️ Toy 5 · Gradient descent scales nudges by the learning rate

A gradient update subtracts `learning_rate × gradient`. Positive gradients move parameters down;
negative gradients move them up.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t5_theta = 2.0
t5_eta = 0.1
t5_grads = np.array([1.0, 0.5, -0.25, 0.0, 0.75, -0.5])
print("gradients:", t5_grads.tolist())          # -> [1.0, 0.5, -0.25, 0.0, 0.75, -0.5]
t5_steps = t5_eta * t5_grads                    # -> [0.1, 0.05, -0.025, 0.0, 0.075, -0.05]
print("eta * gradients:", np.round(t5_steps, 3).tolist())  # -> [0.1, 0.05, -0.025, 0.0, 0.075, -0.05]
t5_updates = -t5_steps                          # -> [-0.1, -0.05, 0.025, -0.0, -0.075, 0.05]
print("signed updates:", np.round(t5_updates, 3).tolist())  # -> [-0.1, -0.05, 0.025, -0.0, -0.075, 0.05]
t5_path = t5_theta + np.cumsum(t5_updates)      # -> [1.9, 1.85, 1.875, 1.875, 1.8, 1.85]
print("theta path:", np.round(t5_path, 3).tolist())  # -> [1.9, 1.85, 1.875, 1.875, 1.8, 1.85]
t5_final = t5_path[-1]                          # -> 1.85
print("final theta:", float(t5_final))          # -> 1.85
assert abs(t5_final - 1.85) < 1e-12

plt.figure(figsize=(4.6, 2.8))
plt.plot(range(1, 7), t5_path, marker="o", color="crimson")
plt.title("Toy 5 · parameter path from small nudges")
plt.xlabel("update")
plt.ylabel("theta")
plt.show()

▶ What you'll see: `theta` moves down, up, or not at all according to the signed scaled gradients.

### ✍️ Toy 6 · Normalization and memory are arithmetic bookkeeping

Normalization rescales activations by their batch statistics, while memory accounting multiplies
saved vectors by width and bytes per float.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t6_acts = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
print("activations:", t6_acts.tolist())         # -> [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
t6_mean = t6_acts.mean()                        # -> 3.5
print("mean:", float(t6_mean))                  # -> 3.5
t6_var = t6_acts.var()                          # -> 2.9166666666666665
print("variance:", float(t6_var))               # -> 2.9166666666666665
t6_normed = (t6_acts - t6_mean) / np.sqrt(t6_var + 1e-5)  # -> [-1.464, -0.878, -0.293, 0.293, 0.878, 1.464]
print("normalized:", np.round(t6_normed, 3).tolist())     # -> [-1.464, -0.878, -0.293, 0.293, 0.878, 1.464]
t6_vectors = 6
t6_width = 128
t6_memory_kb = t6_vectors * t6_width * 4 / 1024 # -> 3.0
print("activation memory KB:", float(t6_memory_kb))        # -> 3.0
assert round(float(t6_memory_kb), 3) == 3.0

plt.figure(figsize=(4.8, 2.8))
plt.bar(range(t6_normed.size), t6_normed, color="purple")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Toy 6 · normalized activation scale")
plt.xlabel("activation index")
plt.ylabel("z-score")
plt.show()

▶ What you'll see: normalized values are centered around zero, and six saved 128-wide vectors cost `3.0 KB`.


## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, vectorized neural-network arithmetic, and numerical checks.
import matplotlib.pyplot as plt  # load Matplotlib for inspecting scores, activations, losses, and learned boundaries.
np.random.seed(0)  # make every random example reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — Compute one affine score

**Goal.** Build the perceptron's weighted sum from scratch, because every MLP layer starts with $z=w^\top x+b$. We build it in 2 steps.

In [ ]:
x_b1 = np.array([1.5, -0.5])  # define a two-feature input.
w_b1 = np.array([1.2, -0.4])  # define one weight per feature.
b_b1 = 0.4  # define the bias term that shifts the threshold.
parts_b1 = x_b1 * w_b1  # compute each feature's contribution before summing.
print("parts:", np.round(parts_b1, 3))  # inspect 1.5*1.2 and -0.5*-0.4.
assert np.allclose(parts_b1, [1.8, 0.2])

▶ What you'll see: both features add positive evidence for this input.

In [ ]:
z_b1 = float(np.dot(w_b1, x_b1) + b_b1)  # add feature evidence and bias.
print("z:", round(z_b1, 3))  # inspect the final affine score.
assert round(z_b1, 3) == 2.400
plt.figure(figsize=(4, 3))
plt.bar(["x0w0", "x1w1", "b"], [parts_b1[0], parts_b1[1], b_b1], color="teal")
plt.title("Basic 1: affine pieces")
plt.ylabel("contribution")
plt.show()

▶ What you'll see: the three bars sum to the affine score 2.4.

👀 Takeaway: a perceptron score is a weighted evidence sum plus a movable threshold.

### Basic 2 — Apply a step decision

**Goal.** Convert a score into a hard binary decision, because the original perceptron predicts by checking which side of a threshold the input lands on. We build it in 2 steps.

In [ ]:
z_b2 = 2.4  # use the lesson score from the affine pass.
threshold_b2 = 0.0  # choose the standard zero threshold.
y_b2 = 1 if z_b2 >= threshold_b2 else 0  # classify by the side of the threshold.
print("score:", z_b2, "decision:", y_b2)
assert y_b2 == 1

▶ What you'll see: a positive score becomes class 1.

In [ ]:
score_grid_b2 = np.linspace(-3, 3, 13)  # create possible perceptron scores.
decisions_b2 = (score_grid_b2 >= threshold_b2).astype(int)  # apply the same hard rule.
plt.figure(figsize=(4.5, 3))
plt.step(score_grid_b2, decisions_b2, where="post", color="darkorange")
plt.axvline(threshold_b2, color="black", linestyle="--")
plt.title("Basic 2: step decision")
plt.xlabel("z")
plt.ylabel("class")
plt.show()

▶ What you'll see: a jump from 0 to 1 at the threshold.

👀 Takeaway: hard perceptron decisions are simple but not smooth enough for gradient-based probability training.

### Basic 3 — Draw a perceptron boundary

**Goal.** Interpret $w^\top x+b=0$ as a line, because linear classifiers separate the plane with a boundary. We build it in 2 steps.

In [ ]:
w_b3 = np.array([1.2, -0.4])  # choose the same two weights.
b_b3 = 0.4  # choose the same bias.
x0_b3 = np.linspace(-1, 2, 80)  # sweep first coordinate values.
x1_boundary_b3 = -(w_b3[0] * x0_b3 + b_b3) / w_b3[1]  # solve w0*x0 + w1*x1 + b = 0.
print("boundary at x0=0:", round(float(-(w_b3[0] * 0 + b_b3) / w_b3[1]), 3))
assert round(float(-(w_b3[0] * 0 + b_b3) / w_b3[1]), 3) == 1.000

▶ What you'll see: when x0 is 0, the boundary crosses x1 at 1.0.

In [ ]:
pts_b3 = np.array([[1.5, -0.5], [-0.2, 1.0], [1.0, 2.0]])  # three points to score.
scores_b3 = pts_b3 @ w_b3 + b_b3  # compute each point's signed distance proxy.
plt.figure(figsize=(4.5, 3.5))
plt.plot(x0_b3, x1_boundary_b3, color="black", label="z=0 boundary")
plt.scatter(pts_b3[:, 0], pts_b3[:, 1], c=scores_b3 > 0, cmap="coolwarm", s=80)
plt.title("Basic 3: perceptron line")
plt.xlabel("x0")
plt.ylabel("x1")
plt.legend()
plt.show()

▶ What you'll see: points on opposite sides of the line receive different colors.

👀 Takeaway: one perceptron can only carve input space with one linear boundary.

### Basic 4 — Gate a score with ReLU

**Goal.** Replace the hard decision with a hidden activation, because MLPs need differentiable-ish signals that can be passed to later layers. We build it in 2 steps.

In [ ]:
z_b4 = np.array([-1.2, 0.0, 0.7, 2.4])  # example affine scores.
h_b4 = np.maximum(0, z_b4)  # ReLU clips negative scores and passes positive scores.
print("z:", z_b4)
print("h:", h_b4)
assert np.allclose(h_b4, [0.0, 0.0, 0.7, 2.4])

▶ What you'll see: ReLU preserves positive scores and blocks negative ones.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.plot(z_b4, h_b4, marker="o", color="seagreen")
plt.axhline(0, color="black", linewidth=0.7)
plt.title("Basic 4: ReLU activation")
plt.xlabel("z")
plt.ylabel("max(0,z)")
plt.show()

▶ What you'll see: the activation is flat for negative inputs and linear for positive inputs.

👀 Takeaway: nonlinear gates are what keep stacked layers from collapsing into one linear model.

### Basic 5 — Compute a hidden layer

**Goal.** Run several perceptrons at once with a matrix multiply, because an MLP hidden layer has many units. We build it in 3 steps.

In [ ]:
x_b5 = np.array([1.5, -0.5])  # one input vector.
W_b5 = np.array([[1.2, -0.4], [-0.7, 1.0], [0.5, 0.5]])  # three hidden weight rows.
b_b5 = np.array([0.4, 0.1, -0.2])  # three hidden biases.
print("W shape:", W_b5.shape, "x shape:", x_b5.shape)
assert W_b5.shape == (3, 2)

▶ What you'll see: a 3×2 matrix maps a length-2 input to three hidden scores.

In [ ]:
z_b5 = W_b5 @ x_b5 + b_b5  # compute all hidden affine scores.
h_b5 = np.maximum(0, z_b5)  # apply ReLU elementwise.
print("z:", np.round(z_b5, 3))
print("h:", np.round(h_b5, 3))
assert np.allclose(np.round(h_b5, 3), [2.4, 0.0, 0.3])

▶ What you'll see: the second unit is shut off by ReLU.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["unit0", "unit1", "unit2"], h_b5, color="purple")
plt.title("Basic 5: hidden activations")
plt.ylabel("h")
plt.show()

▶ What you'll see: the active units are visible as nonzero bars.

👀 Takeaway: a hidden layer is many affine scores gated in parallel.

### Basic 6 — Combine hidden units into an output

**Goal.** Turn hidden activations into one final score, because the output layer weighs the learned features. We build it in 2 steps.

In [ ]:
h_b6 = np.array([2.4, 0.0, 0.3])  # hidden activations from a previous layer.
w2_b6 = np.array([0.8, -0.3, 0.5])  # output weights.
b2_b6 = -0.1  # output bias.
contrib_b6 = h_b6 * w2_b6  # contribution from each hidden feature.
print("output contributions:", np.round(contrib_b6, 3))
assert np.allclose(np.round(contrib_b6, 3), [1.92, -0.0, 0.15])

▶ What you'll see: inactive hidden unit 1 contributes exactly zero.

In [ ]:
yhat_b6 = float(w2_b6 @ h_b6 + b2_b6)  # final scalar output.
print("output score:", round(yhat_b6, 3))
assert round(yhat_b6, 3) == 1.970
plt.figure(figsize=(4.2, 3))
plt.bar(["h0*w0", "h1*w1", "h2*w2", "bias"], [contrib_b6[0], contrib_b6[1], contrib_b6[2], b2_b6], color="teal")
plt.title("Basic 6: output score pieces")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: the final score is mostly driven by the first hidden unit.

👀 Takeaway: output weights decide which hidden features help or hurt the prediction.

### Basic 7 — Turn logits into probabilities

**Goal.** Apply softmax to two class scores, because classifiers need normalized comparisons rather than raw logits. We build it in 2 steps.

In [ ]:
logits_b7 = np.array([2.4, 0.4])  # two class scores.
shifted_b7 = logits_b7 - np.max(logits_b7)  # stable softmax shift.
exp_b7 = np.exp(shifted_b7)  # positive relative weights.
print("shifted logits:", shifted_b7)
print("exp shifted:", np.round(exp_b7, 3))
assert np.allclose(np.round(exp_b7, 3), [1.0, 0.135])

▶ What you'll see: the top class becomes exp(0)=1 after shifting.

In [ ]:
probs_b7 = exp_b7 / np.sum(exp_b7)  # normalize to probabilities.
print("probabilities:", np.round(probs_b7, 3))
assert round(float(probs_b7[0]), 3) == 0.881
plt.figure(figsize=(4, 3))
plt.bar(["class0", "class1"], probs_b7, color="slateblue")
plt.ylim(0, 1)
plt.title("Basic 7: softmax")
plt.ylabel("probability")
plt.show()

▶ What you'll see: the class with logit 2.4 receives about 88.1% probability.

👀 Takeaway: softmax converts score differences into calibrated class probabilities.

### Basic 8 — Compute cross-entropy loss

**Goal.** Measure the penalty for the true class probability, because classification training usually minimizes negative log probability. We build it in 2 steps.

In [ ]:
probs_b8 = np.array([0.881, 0.119])  # probabilities from the softmax example.
y_true_b8 = 0  # the correct class is class 0.
loss_b8 = -np.log(probs_b8[y_true_b8])  # cross-entropy for one example.
print("loss:", round(loss_b8, 3))
assert round(loss_b8, 3) == 0.127

▶ What you'll see: confident correct probability gives a small loss.

In [ ]:
p_grid_b8 = np.linspace(0.05, 0.99, 80)  # possible probabilities assigned to the true class.
loss_grid_b8 = -np.log(p_grid_b8)  # loss curve.
plt.figure(figsize=(4.4, 3))
plt.plot(p_grid_b8, loss_grid_b8, color="crimson")
plt.scatter([probs_b8[y_true_b8]], [loss_b8], color="black")
plt.title("Basic 8: -log(true probability)")
plt.xlabel("p(true class)")
plt.ylabel("loss")
plt.show()

▶ What you'll see: loss falls sharply as the model assigns more probability to the correct class.

👀 Takeaway: cross-entropy rewards confident correct comparisons and heavily punishes confident wrong ones.

### Basic 9 — Take one gradient step

**Goal.** Update one scalar parameter with gradient descent, because neural nets learn by repeated small parameter movements. We build it in 2 steps.

In [ ]:
theta_b9 = 2.0  # current parameter value.
eta_b9 = 0.06  # learning rate.
grad_b9 = 1.35  # local derivative of loss with respect to theta.
move_b9 = eta_b9 * grad_b9  # scaled step.
print("movement:", round(move_b9, 3))
assert round(move_b9, 3) == 0.081

▶ What you'll see: the raw gradient is scaled down before changing the parameter.

In [ ]:
theta_new_b9 = theta_b9 - move_b9  # gradient descent subtracts the scaled gradient.
print("new theta:", round(theta_new_b9, 3))
assert round(theta_new_b9, 3) == 1.919
plt.figure(figsize=(4, 3))
plt.bar(["before", "after"], [theta_b9, theta_new_b9], color=["gray", "seagreen"])
plt.title("Basic 9: one gradient step")
plt.ylabel("theta")
plt.show()

▶ What you'll see: the parameter moves downward because the gradient is positive.

👀 Takeaway: the learning rate converts a local slope into a controlled parameter update.

### Basic 10 — Normalize one activation

**Goal.** Standardize a signal using mean and variance, because scale controls how stable forward values and backward gradients are. We build it in 2 steps.

In [ ]:
value_b10 = 2.4  # activation to normalize.
mean_b10 = 1.0  # local mean.
var_b10 = 0.25  # local variance.
eps_b10 = 0.00001  # small constant to avoid division by zero.
std_b10 = np.sqrt(var_b10 + eps_b10)  # denominator of normalization.
print("std:", round(std_b10, 3))
assert round(std_b10, 3) == 0.500

▶ What you'll see: variance 0.25 corresponds to standard deviation about 0.5.

In [ ]:
normalized_b10 = (value_b10 - mean_b10) / std_b10  # standardize the activation.
print("normalized value:", round(normalized_b10, 3))
assert round(normalized_b10, 3) == 2.800
plt.figure(figsize=(4, 3))
plt.bar(["raw", "mean", "normalized"], [value_b10, mean_b10, normalized_b10], color="darkorange")
plt.title("Basic 10: activation scale")
plt.show()

▶ What you'll see: the raw value is 2.8 standard deviations above the mean.

👀 Takeaway: normalization expresses signals relative to their local scale, not as isolated raw numbers.

## 🟡 Easy

### Easy 1 — Train a perceptron on AND

**Goal.** Use the perceptron update on the AND truth table, because a single linear boundary can represent AND. We build it in 4 steps.

In [ ]:
X_e1 = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])  # AND inputs.
y_e1 = np.array([0., 0., 0., 1.])  # AND targets.
w_e1 = np.zeros(2)  # start with no feature preference.
b_e1 = 0.0  # start threshold at the origin.
print("training examples:", len(X_e1))
assert X_e1.shape == (4, 2)

▶ What you'll see: four binary inputs define the full AND table.

In [ ]:
for epoch_e1 in range(10):  # repeat enough passes for this tiny linearly separable problem.
    errors_e1 = 0
    for xi_e1, yi_e1 in zip(X_e1, y_e1):
        pred_e1 = 1.0 if (w_e1 @ xi_e1 + b_e1) >= 0 else 0.0
        err_e1 = yi_e1 - pred_e1
        w_e1 += 0.2 * err_e1 * xi_e1
        b_e1 += 0.2 * err_e1
        errors_e1 += int(err_e1 != 0)
    if errors_e1 == 0:
        break
print("w:", np.round(w_e1, 3), "b:", round(b_e1, 3), "epochs:", epoch_e1 + 1)

▶ What you'll see: the weights and bias settle into a rule that separates only (1,1) as positive.

In [ ]:
scores_e1 = X_e1 @ w_e1 + b_e1  # compute final scores.
preds_e1 = (scores_e1 >= 0).astype(float)  # hard predictions.
print("predictions:", preds_e1.astype(int))
assert np.array_equal(preds_e1, y_e1)

▶ What you'll see: the perceptron exactly matches `[0, 0, 0, 1]`.

In [ ]:
plt.figure(figsize=(4, 3.5))
plt.scatter(X_e1[:, 0], X_e1[:, 1], c=y_e1, cmap="coolwarm", s=90)
xline_e1 = np.linspace(-0.2, 1.2, 50)
yline_e1 = -(w_e1[0] * xline_e1 + b_e1) / w_e1[1]
plt.plot(xline_e1, yline_e1, color="black")
plt.title("Easy 1: learned AND boundary")
plt.xlabel("x0")
plt.ylabel("x1")
plt.show()

▶ What you'll see: a line that leaves only the top-right point on the positive side.

👀 Takeaway: perceptron updates can learn a linear logic rule by moving the boundary after mistakes.

### Easy 2 — Show why XOR needs hidden units

**Goal.** Compare one line with two hidden ReLU features on XOR, because XOR is the classic reason one perceptron is not enough. We build it in 3 steps.

In [ ]:
X_e2 = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])  # XOR inputs.
y_e2 = np.array([0., 1., 1., 0.])  # XOR targets.
linear_scores_e2 = X_e2 @ np.array([1., 1.]) - 0.5  # one simple linear score.
linear_preds_e2 = (linear_scores_e2 >= 0).astype(float)  # hard line predictions.
print("linear preds:", linear_preds_e2.astype(int))
print("linear mistakes:", int(np.sum(linear_preds_e2 != y_e2)))
assert int(np.sum(linear_preds_e2 != y_e2)) == 1

▶ What you'll see: a single line cannot match all four XOR labels.

In [ ]:
h1_e2 = np.maximum(0, X_e2 @ np.array([1., 1.]) - 0.5)  # activates for at least one 1.
h2_e2 = np.maximum(0, X_e2 @ np.array([1., 1.]) - 1.5)  # activates mainly for two 1s.
score_mlp_e2 = 2 * h1_e2 - 6 * h2_e2 - 0.5  # combine hidden features to isolate exactly-one cases.
pred_mlp_e2 = (score_mlp_e2 >= 0).astype(float)
print("hidden features:\n", np.round(np.c_[h1_e2, h2_e2], 2))
print("MLP preds:", pred_mlp_e2.astype(int))
assert np.array_equal(pred_mlp_e2, y_e2)

▶ What you'll see: two hidden ReLU features make the exactly-one pattern linearly separable in hidden space.

In [ ]:
plt.figure(figsize=(4, 3.5))
plt.scatter(h1_e2, h2_e2, c=y_e2, cmap="coolwarm", s=90)
plt.title("Easy 2: XOR after hidden features")
plt.xlabel("h1")
plt.ylabel("h2")
plt.show()

▶ What you'll see: the XOR labels separate more easily after the nonlinear feature map.

👀 Takeaway: hidden nonlinear units let an MLP represent patterns a single linear boundary cannot.

### Easy 3 — Run a full forward pass for a tiny classifier

**Goal.** Compute hidden scores, activations, logits, probabilities, and loss for one example, because these are the forward-pass objects backprop uses. We build it in 4 steps.

In [ ]:
x_e3 = np.array([1.5, -0.5])
W1_e3 = np.array([[1.2, -0.4], [-0.7, 1.0], [0.5, 0.5]])
b1_e3 = np.array([0.4, 0.1, -0.2])
W2_e3 = np.array([[0.8, -0.3, 0.5], [-0.2, 0.4, -0.1]])
b2_e3 = np.array([-0.1, 0.1])
print("shapes:", W1_e3.shape, W2_e3.shape)
assert W2_e3.shape == (2, 3)

▶ What you'll see: the output layer expects the three hidden activations.

In [ ]:
z1_e3 = W1_e3 @ x_e3 + b1_e3
h_e3 = np.maximum(0, z1_e3)
print("z1:", np.round(z1_e3, 3), "h:", np.round(h_e3, 3))
assert np.allclose(np.round(h_e3, 3), [2.4, 0.0, 0.3])

▶ What you'll see: ReLU turns one negative hidden score into zero.

In [ ]:
logits_e3 = W2_e3 @ h_e3 + b2_e3
prob_e3 = np.exp(logits_e3 - np.max(logits_e3))
prob_e3 = prob_e3 / np.sum(prob_e3)
loss_e3 = -np.log(prob_e3[0])
print("logits:", np.round(logits_e3, 3), "prob:", np.round(prob_e3, 3), "loss:", round(loss_e3, 3))
assert round(float(logits_e3[0]), 3) == 1.970

▶ What you'll see: the network produces two logits, probabilities, and a class-0 loss.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["logit0", "logit1"], logits_e3, color="steelblue")
plt.title("Easy 3: classifier logits")
plt.ylabel("score")
plt.show()

▶ What you'll see: class 0 has the larger score, so its probability is larger.

👀 Takeaway: a forward pass is a sequence of cached intermediate values used later by gradients.

### Easy 4 — Backpropagate through one output layer

**Goal.** Compute softmax-cross-entropy gradients for the output layer, because the famous simplification is gradient = probabilities − one-hot target. We build it in 4 steps.

In [ ]:
h_e4 = np.array([2.4, 0.0, 0.3])  # hidden activations.
logits_e4 = np.array([1.97, -0.41])  # two output logits.
y_e4 = np.array([1.0, 0.0])  # class 0 is correct.
prob_e4 = np.exp(logits_e4 - np.max(logits_e4))
prob_e4 = prob_e4 / np.sum(prob_e4)
print("prob:", np.round(prob_e4, 3))
assert round(float(prob_e4[0]), 3) == 0.915

▶ What you'll see: class 0 is predicted with high probability.

In [ ]:
dlogits_e4 = prob_e4 - y_e4  # gradient of CE loss with respect to logits.
print("dlogits:", np.round(dlogits_e4, 3))
assert round(float(np.sum(dlogits_e4)), 6) == 0.0

▶ What you'll see: the correct class has a negative gradient and the wrong class has a positive gradient.

In [ ]:
dW2_e4 = dlogits_e4[:, None] * h_e4[None, :]  # outer product gives output-weight gradients.
db2_e4 = dlogits_e4.copy()
print("dW2:\n", np.round(dW2_e4, 3))
print("db2:", np.round(db2_e4, 3))
assert dW2_e4.shape == (2, 3)

▶ What you'll see: inactive hidden unit 1 has zero gradient column because h1 is zero.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.imshow(dW2_e4, cmap="coolwarm", aspect="auto")
plt.colorbar(label="gradient")
plt.title("Easy 4: output weight gradients")
plt.xlabel("hidden unit")
plt.ylabel("class")
plt.show()

▶ What you'll see: the two class rows have opposite-signed gradients.

👀 Takeaway: output-layer gradients are simple outer products of class error and hidden activation.

### Easy 5 — Train a one-hidden-layer MLP on XOR

**Goal.** Fit XOR with NumPy backprop, because it demonstrates how hidden nonlinear units learn a nonlinearly separable rule. We build it in 5 steps.

In [ ]:
X_e5 = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_e5 = np.array([[0.], [1.], [1.], [0.]])
rng_e5 = np.random.default_rng(5)
W1_e5 = rng_e5.normal(scale=0.8, size=(2, 3))
b1_e5 = np.zeros((1, 3))
W2_e5 = rng_e5.normal(scale=0.8, size=(3, 1))
b2_e5 = np.zeros((1, 1))
print("parameter shapes:", W1_e5.shape, W2_e5.shape)

▶ What you'll see: a 2→3→1 MLP has a small set of trainable arrays.

In [ ]:
losses_e5 = []
for step_e5 in range(5000):
    z1_e5 = X_e5 @ W1_e5 + b1_e5
    h_e5 = np.tanh(z1_e5)
    logits_e5 = h_e5 @ W2_e5 + b2_e5
    p_e5 = 1 / (1 + np.exp(-logits_e5))
    loss_e5 = float(np.mean(-(y_e5 * np.log(p_e5 + 1e-9) + (1 - y_e5) * np.log(1 - p_e5 + 1e-9))))
    dlogits_e5 = (p_e5 - y_e5) / len(X_e5)
    dW2_e5 = h_e5.T @ dlogits_e5
    db2_e5 = np.sum(dlogits_e5, axis=0, keepdims=True)
    dh_e5 = dlogits_e5 @ W2_e5.T
    dz1_e5 = dh_e5 * (1 - h_e5 ** 2)
    dW1_e5 = X_e5.T @ dz1_e5
    db1_e5 = np.sum(dz1_e5, axis=0, keepdims=True)
    W2_e5 -= 0.3 * dW2_e5; b2_e5 -= 0.3 * db2_e5
    W1_e5 -= 0.3 * dW1_e5; b1_e5 -= 0.3 * db1_e5
    if step_e5 % 100 == 0:
        losses_e5.append(loss_e5)
print("loss start -> end:", round(losses_e5[0], 3), "->", round(losses_e5[-1], 3))
assert losses_e5[-1] < losses_e5[0]

▶ What you'll see: the cross-entropy loss decreases over training.

In [ ]:
z1_e5 = X_e5 @ W1_e5 + b1_e5
h_e5 = np.tanh(z1_e5)
p_e5 = 1 / (1 + np.exp(-(h_e5 @ W2_e5 + b2_e5)))
preds_e5 = (p_e5 >= 0.5).astype(int)
print("probabilities:", np.round(p_e5.ravel(), 3))
print("predictions:", preds_e5.ravel())
assert np.array_equal(preds_e5, y_e5.astype(int))

▶ What you'll see: the trained MLP predicts XOR correctly.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(np.arange(len(losses_e5)) * 100, losses_e5, color="teal")
plt.title("Easy 5: XOR MLP training loss")
plt.xlabel("step")
plt.ylabel("cross-entropy")
plt.show()

▶ What you'll see: the loss curve drops as hidden features and output weights coordinate.

In [ ]:
plt.figure(figsize=(4, 3.5))
plt.scatter(h_e5[:, 0], h_e5[:, 1], c=y_e5.ravel(), cmap="coolwarm", s=90)
plt.title("Easy 5: learned hidden space")
plt.xlabel("h0")
plt.ylabel("h1")
plt.show()

▶ What you'll see: XOR points are rearranged into a hidden representation the output can separate.

👀 Takeaway: backprop lets hidden layers learn feature maps that make hard input patterns easier.

## 🔴 Advanced

### Advanced 1 — Visualize a learned nonlinear decision surface

**Goal.** Train a small tanh MLP on circular data and plot its probability surface, because multilayer networks create curved boundaries by composing simple maps. We build it in 5 steps.

In [ ]:
rng_a1 = np.random.default_rng(11)
angles_a1 = np.linspace(0, 2 * np.pi, 80, endpoint=False)
radii_a1 = np.r_[np.full(40, 0.6), np.full(40, 1.4)]
angles_full_a1 = np.r_[angles_a1[:40], angles_a1[:40]]
X_a1 = np.c_[radii_a1 * np.cos(angles_full_a1), radii_a1 * np.sin(angles_full_a1)]
y_a1 = np.r_[np.zeros(40), np.ones(40)][:, None]
print("class counts:", np.bincount(y_a1.ravel().astype(int)))
assert X_a1.shape == (80, 2)

▶ What you'll see: two rings form a nonlinear classification problem.

In [ ]:
W1_a1 = rng_a1.normal(scale=0.7, size=(2, 8)); b1_a1 = np.zeros((1, 8))
W2_a1 = rng_a1.normal(scale=0.7, size=(8, 1)); b2_a1 = np.zeros((1, 1))
losses_a1 = []
for step_a1 in range(2500):
    z1_a1 = X_a1 @ W1_a1 + b1_a1
    h_a1 = np.tanh(z1_a1)
    logit_a1 = h_a1 @ W2_a1 + b2_a1
    p_a1 = 1 / (1 + np.exp(-logit_a1))
    loss_a1 = float(np.mean(-(y_a1 * np.log(p_a1 + 1e-9) + (1 - y_a1) * np.log(1 - p_a1 + 1e-9))))
    dlogit_a1 = (p_a1 - y_a1) / len(X_a1)
    dW2_a1 = h_a1.T @ dlogit_a1; db2_a1 = np.sum(dlogit_a1, axis=0, keepdims=True)
    dz1_a1 = (dlogit_a1 @ W2_a1.T) * (1 - h_a1 ** 2)
    dW1_a1 = X_a1.T @ dz1_a1; db1_a1 = np.sum(dz1_a1, axis=0, keepdims=True)
    W2_a1 -= 0.2 * dW2_a1; b2_a1 -= 0.2 * db2_a1
    W1_a1 -= 0.2 * dW1_a1; b1_a1 -= 0.2 * db1_a1
    if step_a1 % 100 == 0:
        losses_a1.append(loss_a1)
print("loss start -> end:", round(losses_a1[0], 3), "->", round(losses_a1[-1], 3))
assert losses_a1[-1] < 0.1

▶ What you'll see: training drives the ring-classification loss close to zero.

In [ ]:
grid_x_a1, grid_y_a1 = np.meshgrid(np.linspace(-1.8, 1.8, 120), np.linspace(-1.8, 1.8, 120))
grid_a1 = np.c_[grid_x_a1.ravel(), grid_y_a1.ravel()]
h_grid_a1 = np.tanh(grid_a1 @ W1_a1 + b1_a1)
p_grid_a1 = 1 / (1 + np.exp(-(h_grid_a1 @ W2_a1 + b2_a1)))
print("grid probability range:", round(float(p_grid_a1.min()), 3), round(float(p_grid_a1.max()), 3))
assert p_grid_a1.max() > 0.9 and p_grid_a1.min() < 0.1

▶ What you'll see: the trained model assigns low and high probabilities across the plane.

In [ ]:
plt.figure(figsize=(4.6, 4))
plt.contourf(grid_x_a1, grid_y_a1, p_grid_a1.reshape(grid_x_a1.shape), levels=20, cmap="coolwarm", alpha=0.7)
plt.scatter(X_a1[:, 0], X_a1[:, 1], c=y_a1.ravel(), cmap="coolwarm", edgecolor="black", s=35)
plt.title("Advanced 1: nonlinear MLP boundary")
plt.xlabel("x0")
plt.ylabel("x1")
plt.show()

▶ What you'll see: the decision surface bends around the inner ring rather than forming one straight line.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.plot(np.arange(len(losses_a1)) * 100, losses_a1, color="purple")
plt.title("Advanced 1: training curve")
plt.xlabel("step")
plt.ylabel("loss")
plt.show()

▶ What you'll see: the loss curve confirms the visible boundary came from optimization, not hand drawing.

👀 Takeaway: MLPs compose simple hidden units into curved decision regions.

### Advanced 2 — Watch vanishing gradients through sigmoid depth

**Goal.** Multiply derivatives through a deep stack, because backprop signals can shrink when each layer contributes a factor below one. We build it in 3 steps.

In [ ]:
depths_a2 = np.arange(1, 16)  # stack depths to inspect.
sigmoid_slope_a2 = 0.25  # maximum derivative of sigmoid occurs at zero.
relu_slope_a2 = 1.0  # active ReLU derivative is one.
sigmoid_grad_a2 = sigmoid_slope_a2 ** depths_a2  # worst-case repeated shrinkage.
relu_grad_a2 = relu_slope_a2 ** depths_a2  # active ReLU preserves magnitude in this toy comparison.
print("sigmoid grad at depth 10:", round(float(sigmoid_grad_a2[9]), 8))
assert round(float(sigmoid_grad_a2[9]), 8) == 0.00000095

▶ What you'll see: multiplying 0.25 ten times nearly erases the gradient.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.semilogy(depths_a2, sigmoid_grad_a2, marker="o", label="sigmoid max slope")
plt.semilogy(depths_a2, relu_grad_a2, marker="s", label="active ReLU slope")
plt.title("Advanced 2: gradient products")
plt.xlabel("depth")
plt.ylabel("backprop multiplier (log)")
plt.legend()
plt.show()

▶ What you'll see: the sigmoid curve falls exponentially on a log scale.

In [ ]:
preacts_a2 = np.linspace(-6, 6, 200)
sig_a2 = 1 / (1 + np.exp(-preacts_a2))
deriv_a2 = sig_a2 * (1 - sig_a2)
print("max sigmoid derivative:", round(float(deriv_a2.max()), 3))
assert round(float(deriv_a2.max()), 3) == 0.250
plt.figure(figsize=(4.6, 3))
plt.plot(preacts_a2, deriv_a2, color="crimson")
plt.title("Advanced 2: sigmoid derivative")
plt.xlabel("preactivation")
plt.ylabel("slope")
plt.show()

▶ What you'll see: sigmoid only has slope near 0.25 around zero and even smaller slopes in saturation.

👀 Takeaway: deep training depends on preserving gradient scale, not just choosing expressive layers.

### Advanced 3 — Compare learning rates on the same MLP

**Goal.** Train identical models with different learning rates, because step size changes how quickly the same gradients become useful progress. We build it in 4 steps.

In [ ]:
X_a3 = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_a3 = np.array([[0.], [1.], [1.], [0.]])
rates_a3 = [0.03, 0.3, 3.0]
curves_a3 = []
print("learning rates:", rates_a3)
assert len(rates_a3) == 3

▶ What you'll see: three step sizes will be compared on the same XOR task.

In [ ]:
for eta_a3 in rates_a3:
    rng_a3 = np.random.default_rng(21)
    W1_a3 = rng_a3.normal(scale=0.8, size=(2, 4)); b1_a3 = np.zeros((1, 4))
    W2_a3 = rng_a3.normal(scale=0.8, size=(4, 1)); b2_a3 = np.zeros((1, 1))
    losses_one_a3 = []
    for step_a3 in range(800):
        z1_a3 = X_a3 @ W1_a3 + b1_a3; h_a3 = np.tanh(z1_a3)
        p_a3 = 1 / (1 + np.exp(-(h_a3 @ W2_a3 + b2_a3)))
        loss_a3 = float(np.mean(-(y_a3 * np.log(p_a3 + 1e-9) + (1 - y_a3) * np.log(1 - p_a3 + 1e-9))))
        dlogit_a3 = (p_a3 - y_a3) / len(X_a3)
        dW2_a3 = h_a3.T @ dlogit_a3; db2_a3 = np.sum(dlogit_a3, axis=0, keepdims=True)
        dz1_a3 = (dlogit_a3 @ W2_a3.T) * (1 - h_a3 ** 2)
        dW1_a3 = X_a3.T @ dz1_a3; db1_a3 = np.sum(dz1_a3, axis=0, keepdims=True)
        W2_a3 -= eta_a3 * dW2_a3; b2_a3 -= eta_a3 * db2_a3
        W1_a3 -= eta_a3 * dW1_a3; b1_a3 -= eta_a3 * db1_a3
        if step_a3 % 20 == 0:
            losses_one_a3.append(loss_a3)
    curves_a3.append(losses_one_a3)
print("final losses:", [round(c[-1], 3) for c in curves_a3])
assert curves_a3[1][-1] < curves_a3[0][-1]

▶ What you'll see: the moderate learning rate improves faster than the tiny one.

In [ ]:
plt.figure(figsize=(5, 3))
for eta_a3, curve_a3 in zip(rates_a3, curves_a3):
    plt.plot(np.arange(len(curve_a3)) * 20, curve_a3, label=f"eta={eta_a3}")
plt.title("Advanced 3: learning-rate sweep")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

▶ What you'll see: different step sizes produce visibly different training curves.

In [ ]:
best_idx_a3 = int(np.argmin([c[-1] for c in curves_a3]))
print("best displayed eta:", rates_a3[best_idx_a3])
assert curves_a3[best_idx_a3][-1] <= min(c[-1] for c in curves_a3)

▶ What you'll see: the sweep identifies which displayed learning rate reached the lowest final loss.

👀 Takeaway: learning rate is the knob that turns gradient information into faster or slower progress, and it must be tuned rather than assumed.

### Advanced 4 — Measure parameter and activation memory

**Goal.** Count parameters and activations for a small MLP, because shape arithmetic becomes hardware cost during training. We build it in 3 steps.

In [ ]:
batch_a4 = 32
sizes_a4 = np.array([64, 128, 10])  # input, hidden, output widths.
param_count_a4 = sizes_a4[0] * sizes_a4[1] + sizes_a4[1] + sizes_a4[1] * sizes_a4[2] + sizes_a4[2]
param_kb_a4 = param_count_a4 * 4 / 1024
print("parameters:", int(param_count_a4), "param KB:", round(float(param_kb_a4), 3))
assert int(param_count_a4) == 9610

▶ What you'll see: even a small dense MLP has thousands of parameters.

In [ ]:
act_count_a4 = batch_a4 * (sizes_a4[0] + sizes_a4[1] + sizes_a4[2])
act_kb_a4 = act_count_a4 * 4 / 1024
print("activation KB:", round(float(act_kb_a4), 3))
assert round(float(act_kb_a4), 3) == 25.250

▶ What you'll see: saved activations for a batch can be comparable to or larger than parameter storage.

In [ ]:
batches_a4 = np.array([1, 8, 32, 128, 512])
act_kb_curve_a4 = batches_a4 * (sizes_a4.sum()) * 4 / 1024
plt.figure(figsize=(4.8, 3))
plt.plot(batches_a4, act_kb_curve_a4, marker="o", color="darkorange")
plt.title("Advanced 4: activation memory vs batch")
plt.xlabel("batch size")
plt.ylabel("activation KB")
plt.show()

▶ What you'll see: activation memory grows linearly as batch size increases.

👀 Takeaway: MLP math must be tracked with shapes because training stores intermediate activations for the backward pass.

### Advanced 5 — Add L2 regularization to control capacity

**Goal.** Train two MLPs on noisy data with and without L2 penalty, because regularization constrains large weights when capacity could chase noise. We build it in 5 steps.

In [ ]:
rng_a5 = np.random.default_rng(31)
X_a5 = rng_a5.normal(size=(80, 2))
clean_a5 = (X_a5[:, 0] * X_a5[:, 1] > 0).astype(float)[:, None]
flip_a5 = rng_a5.choice(80, size=8, replace=False)
y_a5 = clean_a5.copy()
y_a5[flip_a5] = 1 - y_a5[flip_a5]
print("noisy positives:", int(y_a5.sum()), "flipped labels:", len(flip_a5))
assert len(flip_a5) == 8

▶ What you'll see: the dataset contains deliberately flipped labels.

In [ ]:
lams_a5 = [0.0, 0.1]
train_loss_a5 = []
weight_norm_a5 = []
for lam_a5 in lams_a5:
    rng_model_a5 = np.random.default_rng(41)
    W1_a5 = rng_model_a5.normal(scale=0.5, size=(2, 10)); b1_a5 = np.zeros((1, 10))
    W2_a5 = rng_model_a5.normal(scale=0.5, size=(10, 1)); b2_a5 = np.zeros((1, 1))
    for step_a5 in range(1200):
        z1_a5 = X_a5 @ W1_a5 + b1_a5; h_a5 = np.tanh(z1_a5)
        p_a5 = 1 / (1 + np.exp(-(h_a5 @ W2_a5 + b2_a5)))
        dlogit_a5 = (p_a5 - y_a5) / len(X_a5)
        dW2_a5 = h_a5.T @ dlogit_a5 + lam_a5 * W2_a5
        db2_a5 = np.sum(dlogit_a5, axis=0, keepdims=True)
        dz1_a5 = (dlogit_a5 @ W2_a5.T) * (1 - h_a5 ** 2)
        dW1_a5 = X_a5.T @ dz1_a5 + lam_a5 * W1_a5
        db1_a5 = np.sum(dz1_a5, axis=0, keepdims=True)
        W2_a5 -= 0.2 * dW2_a5; b2_a5 -= 0.2 * db2_a5
        W1_a5 -= 0.2 * dW1_a5; b1_a5 -= 0.2 * db1_a5
    p_final_a5 = 1 / (1 + np.exp(-(np.tanh(X_a5 @ W1_a5 + b1_a5) @ W2_a5 + b2_a5)))
    ce_a5 = float(np.mean(-(y_a5 * np.log(p_final_a5 + 1e-9) + (1 - y_a5) * np.log(1 - p_final_a5 + 1e-9))))
    norm_a5 = float(np.sqrt(np.sum(W1_a5 ** 2) + np.sum(W2_a5 ** 2)))
    train_loss_a5.append(ce_a5); weight_norm_a5.append(norm_a5)
print("losses:", np.round(train_loss_a5, 3))
print("weight norms:", np.round(weight_norm_a5, 3))
assert weight_norm_a5[1] < weight_norm_a5[0]

▶ What you'll see: L2 regularization produces a smaller weight norm.

In [ ]:
plt.figure(figsize=(4.7, 3))
plt.bar(["no L2", "L2=0.1"], weight_norm_a5, color=["crimson", "seagreen"])
plt.title("Advanced 5: regularization shrinks weights")
plt.ylabel("combined weight norm")
plt.show()

▶ What you'll see: the regularized model has visibly smaller weights.

In [ ]:
plt.figure(figsize=(4.7, 3))
plt.bar(["no L2", "L2=0.1"], train_loss_a5, color=["gray", "teal"])
plt.title("Advanced 5: fit vs constraint")
plt.ylabel("training cross-entropy")
plt.show()

▶ What you'll see: regularization may accept a higher training loss to keep the function simpler.

In [ ]:
ratio_a5 = weight_norm_a5[1] / weight_norm_a5[0]
print("regularized/unregularized norm ratio:", round(ratio_a5, 3))
assert ratio_a5 < 0.8

▶ What you'll see: the regularized weights are a clear fraction of the unregularized size.

👀 Takeaway: L2 is a capacity constraint — it trades some training fit for smaller, often more stable functions.